## 0. Setup and Initialization
Before diving into use cases, we should initialize the notebook with the required setup:

In [ ]:
# Put the helpers_root directory itself on sys.path.
import sys
from pathlib import Path

HELPERS_ROOT = Path("/Users/indro/src/tutorials1/helpers_root")
if str(HELPERS_ROOT) not in sys.path:
    sys.path.insert(0, str(HELPERS_ROOT))


In [ ]:
# Import the helper script.
import helpers.hllm as hllm
import pandas as pd
# Set up logging for debugging.
import logging
logging.basicConfig(level=logging.INFO)

# Set OpenAI API key.
import os
from typing import List, Tuple

In [ ]:
os.environ["OPENAI_API_KEY"] = "<your_api_key_here>"

## 1. Travel Agent chat assistant: 
#### Goal: Cretae a chat agent that will help the user to create an itinary to visit New York Trip considering all the constraints.

In [18]:
# Define the prompt for the travel assistant
user_prompt = """
I am visiting New York City for 3 days. Please create a detailed itinerary, 
including popular attractions, food recommendations, and some evening activities.
I already booked flight tickets and hotel near Newark penn station.
Constraints:
1) Dates: from 24th to 27th Dec.
1) My budget for travel is around $400 excluding hotel and flight.
2) I am planning to travel through subway and for rest of the trip I am planning to walk.
3) Also, take into account traffic and tourist rush at popular places.
"""

# Define the system instructions for the assistant
system_instructions = """
You are a travel assistant specializing in creating personalized travel itineraries.
Your recommendations should balance sightseeing, food, and leisure activities considering provided constraints.
Provide details like the time required for activities and approximate costs where possible.
"""

# Use the get_completion method to generate the trip plan
trip_plan = hllm.get_completion(
    user_prompt=user_prompt,
    system_prompt=system_instructions,
    model="gpt-4o-mini",
    temperature=0.7  # Slightly increase temperature for creative outputs
)

# Print the generated trip itinerary
print("3-Day New York City Trip Itinerary:")
print(trip_plan)

LLM API call ... 


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


LLM API call done (40.372 s)
3-Day New York City Trip Itinerary:
Here’s a detailed 3-day itinerary for your trip to New York City from December 24th to December 27th. This plan includes a mix of popular attractions, food recommendations, and evening activities, all while keeping your budget in mind.

### Day 1: December 24th (Christmas Eve)

**Morning:**
- **Breakfast at a Local Café** (8:00 AM - 9:00 AM)
  - Try a bagel with cream cheese and a coffee at a nearby café like “Brooklyn Bagel & Coffee Company.” (~$10)
  
- **Visit Central Park** (9:30 AM - 12:00 PM)
  - Walk around the park, visit Bethesda Terrace, and enjoy the winter scenery. (Free)

**Lunch:**
- **Lunch at The Loeb Boathouse** (12:30 PM - 1:30 PM)
  - A lovely spot overlooking the lake. Grab a light meal, like a salad or sandwich. (~$25)

**Afternoon:**
- **Visit The Metropolitan Museum of Art** (2:00 PM - 5:00 PM)
  - Suggested donation is $25, but you can pay what you wish. Spend a few hours exploring the museum’s vas

## 2. Batch Upload to Vector Store and Query
#### Goal: Add multiple files to a vector store for RAG

In [ ]:
# Upload files to a vector store.
vector_store_name = "batch_vector_store"
file_paths = [
    "/Users/indro/src/tutorials1/helpers_root/docs/tools/all.imports_and_packages.how_to_guide.md", 
    "/Users/indro/src/tutorials1/helpers_root/docs/tools/unit_test/all.write_unit_tests.how_to_guide.md",
             "/Users/indro/src/tutorials1/helpers_root/docs/code_guidelines/all.coding_style.how_to_guide.md"]  # Example paths

question = "Is `from pathlib import Path` a correct import according to the coding guidelines?"

# Create or find vector store.
llm = hllm.LLMClient(model="gpt-4o")
llm.create_client()
client = llm.client
vector_store = client.vector_stores.create(name=vector_store_name)

# Upload files to the vector store.
file_streams = [open(path, "rb") for path in file_paths]
file_batch = client.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vector_store.id, files=file_streams
)

if file_batch.status != "completed" or file_batch.file_counts.failed > 0:
    raise RuntimeError(f"Ingestion not ready: status={file_batch.status}, counts={file_batch.counts}")

resp = client.responses.create(
    model="gpt-4o",
    input=question,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id],
    }],
)

# Extract the assistant's text.
out_text = getattr(resp, "output_text", "")

# Best-effort extraction of cited sources from output annotations.
sources: List[Tuple[str, str]] = []
for item in getattr(resp, "output", []) or []:
    for part in getattr(item, "content", []) or []:
        if getattr(part, "type", "") == "output_text":
            annotations = getattr(getattr(part, "text", None), "annotations", []) or []
            for ann in annotations:
                if getattr(ann, "type", "") == "file_citation":
                    file_id = ann.file_citation.file_id
                    fobj = client.files.retrieve(file_id)
                    sources.append((fobj.filename, file_id))


# Display file batch status
print("\n=== ANSWER ===\n", out_text)

if sources:
    print("\n=== SOURCES ===")
    for name, fid in sources:
        print(f"- {name}  ({fid})")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/vector_stores "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/files "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/files "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/files "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/vector_stores/vs_68ba951b148481918c096d0bf4b1965e/file_batches "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.openai.com/v1/vector_stores/vs_68ba951b148481918c096d0bf4b1965e/file_batches/vsfb_aad13b11299f482f907b36e69430b020 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.openai.com/v1/vector_stores/vs_68ba951b148481918c096d0bf4b1965e/file_batches/vsfb_aad13b11299f482f907b36e69430b020 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.openai.com/v1/vector_stores/vs_68ba951b148481918c096d0bf4b1965e/file_batches/vsfb_aad13b11299f482f907b36e69430b020 "HTTP/1.1 200 OK"
INFO:httpx:HT


=== ANSWER ===
 According to the coding guidelines, using `from pathlib import Path` is not recommended. The guideline suggests starting with `import`, such as `import library.sublibrary as short_name`, to avoid maintenance issues, potential name collisions, and debugging difficulties.


## 3. Apply Prompt to DataFrame
#### Goal: Run prompts batch-wise on a lot of data

In [ ]:
df = pd.DataFrame({"question": [
    "Summarize: Attention is all you need.",
    "Summarize: Diffusion models in 2 sentences.",
    "Summarize: Convnets vs Transformers for vision.",
]})

df_out = hllm.apply_prompt_to_dataframe(
    df=df,
    prompt="Summarize each item in one sentence.",
    model="gpt-4o-mini",
    input_col="question",
    response_col="summary",
    chunk_size=3,
    allow_overwrite=True, 
)
print(df_out.head())

Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

LLM API call ... 


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Processing chunks: 100%|██████████| 1/1 [00:05<00:00,  5.46s/it]

LLM API call done (5.404 s)
                                          question  \
0            Summarize: Attention is all you need.   
1      Summarize: Diffusion models in 2 sentences.   
2  Summarize: Convnets vs Transformers for vision.   

                                             summary  
0  "Attention is All You Need" introduces the Tra...  
1  Diffusion models are generative models that le...  
2  Convnets (Convolutional Neural Networks) excel...  


## 3. Use the Cost Tracker
#### Goal: Find the cumulative or individual costs of your LLM jobs

In [ ]:
tracker = hllm.LLMCostTracker()
txt = hllm.get_completion(
    "Say hello in 10 words.",
    system_prompt="You are terse.",
    model="gpt-4o-mini",
    cache_mode="NORMAL",
    temperature=0.1,
    max_tokens=1000,
    print_cost=True,            
    cost_tracker=tracker,       
)

txt2 = hllm.get_completion(
    "Say hello in 50 words.",
    system_prompt="You are terse.",
    model="gpt-4o-mini",
    cache_mode="NORMAL",
    temperature=0.1,
    max_tokens=1000,
    print_cost=True,         
    cost_tracker=tracker,       
)

txt3 = hllm.get_completion(
    "Say hello in 70 words.",
    system_prompt="You are terse.",
    model="gpt-4o-mini",
    cache_mode="NORMAL",
    temperature=0.1,
    max_tokens=1000,
    print_cost=True,           
    cost_tracker=tracker,      
)

print("Custom tracker total: $", tracker.get_current_cost())
tracker.end_logging_costs()

LLM API call ... 


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


LLM API call done (3.246 s)
cost=$0.000010
LLM API call ... 


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


LLM API call done (3.851 s)
cost=$0.000040
LLM API call ... 


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


LLM API call done (2.298 s)
cost=$0.000054
Custom tracker total: $ 0.00010409999999999998
